# 07 — Compose the workflow

Trace typed dataflow, scatter the crop over two bands and collect the declared output Directory.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/07-compose-the-workflow/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/07-compose-the-workflow/lesson.md) · [Exercise](../modules/07-compose-the-workflow/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/07-compose-the-workflow"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Trace the workflow graph

Predict how many jobs run: two crops, one normalized difference, one Otsu and one STAC. `crop/cropped` becomes an ordered `File[]` because the crop step scatters over `bands`.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
import os
import yaml
contract = yaml.safe_load((Path(os.environ['WORKSPACE']) / 'reference/waterbodies.cwl').read_text())
workflow = next(process for process in contract['$graph'] if process['class'] == 'Workflow')
print(yaml.safe_dump(workflow, sort_keys=False))
assert workflow['steps']['crop']['scatter'] == 'band'
assert workflow['steps']['norm-diff']['in']['rasters'] == 'crop/cropped'
PYTHON
)


## Supply runnable inputs and execute

The job file uses an absolute staged Directory path so moving it into scratch space does not break input resolution. The runner uses the same installed commands as previous labs. Logs and output JSON remain beside the notebook results.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
from scripts.fixtures import create_fixture
create_fixture(Path('source'))
print('Created an 8 x 8 STAC scene with green and NIR assets under source/')
PYTHON

python - <<'PYTHON'
from pathlib import Path
import yaml
job = {'item': {'class': 'Directory', 'path': str(Path('source').resolve())},
       'aoi': '1,1,7,7', 'epsg': 'EPSG:4326', 'bands': ['green', 'nir']}
Path('inputs.yaml').write_text(yaml.safe_dump(job))
PYTHON
run_directory=$(mktemp -d "$LAB_DIR/workflow.XXXXXX")
cwltool --no-container --outdir "$run_directory" "$WORKSPACE/reference/waterbodies.cwl#waterbodies" inputs.yaml > results.json 2> runner.log
cat runner.log
cat results.json
)


## Verify graph execution and its collected product

Find the two crop jobs in the displayed runner log and trace green/NIR order. Then validate the real catalog and count mask pixels.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
import numpy as np
import rasterio
from waterbodies.common import read_item
from waterbodies.validation import validate_item
import json
from urllib.parse import unquote, urlparse
product = json.loads(Path('results.json').read_text())['stac-catalog']
catalog_path = Path(unquote(urlparse(product['location']).path))
item = read_item(catalog_path)
validate_item(item)
href = item.assets['data'].get_absolute_href()
assert href is not None
with rasterio.open(href) as raster:
    pixels = raster.read(1)
    print('Grid:', raster.shape, 'Water pixels:', np.count_nonzero(pixels))
    assert raster.shape == (6, 6)
    assert np.count_nonzero(pixels) == 18
print('STAC validation passed:', item.id)
PYTHON

python -m pytest "$WORKSPACE/tests/workflow"
)


## Record your findings

**Expected outcome:** The runner executes five tool jobs, preserves green/NIR ordering and collects a validated STAC Directory with 18 water pixels.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/07-compose-the-workflow/solution.md) after the activity. Return to [the lab index](README.md).
